# GeoTIFF Pipeline: COG Generation (Multi-Dataset)

Generates Cloud-Optimized GeoTIFF (COG) files from `climate_gt` data for a specific dataset.

**Output:** `local_only/climate_gt/{dataset_id}/{variable}/{time}/z{0-5}.tif`

5-band COGs: RGB (colormapped) + Grayscale (normalized) + Alpha (transparency)

## Setup & Configuration

In [ ]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
import rasterio
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS
from pyproj import Transformer
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))
from app.utils.colormap_utils import apply_colormap, DEFAULT_COLORMAP, list_colormaps

print(f"Libraries imported — default colormap: {DEFAULT_COLORMAP}")

In [ ]:
# Database configuration
DB_PARAMS = {
    "host": "localhost",
    "user": "leon",
    "password": "leon",
    "database": "oc-database",
    "port": 5432,
}

# Australia bounds (WGS84)
AUSTRALIA_BOUNDS_WGS84 = {
    "west": 112.900000,
    "south": -43.650000,
    "east": 153.650000,
    "north": -10.050000,
}

# Web Mercator bounds
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
minx_wm, miny_wm = transformer.transform(AUSTRALIA_BOUNDS_WGS84['west'], AUSTRALIA_BOUNDS_WGS84['south'])
maxx_wm, maxy_wm = transformer.transform(AUSTRALIA_BOUNDS_WGS84['east'], AUSTRALIA_BOUNDS_WGS84['north'])
AUSTRALIA_BOUNDS_WEB_MERCATOR = {"west": minx_wm, "south": miny_wm, "east": maxx_wm, "north": maxy_wm}

# Zoom levels
ZOOM_SPECS = {
    0: (256, 256),
    1: (512, 512),
    2: (1024, 1024),
    3: (2048, 2048),
    4: (4096, 4096),
    5: (8192, 8192),
}

COLORMAP_NAME = DEFAULT_COLORMAP

print(f"Database: {DB_PARAMS['database']}")
print(f"Colormap: {COLORMAP_NAME}")
print(f"Bounds (WGS84): {AUSTRALIA_BOUNDS_WGS84}")

## Select Dataset

In [ ]:
# List available datasets
conn = psycopg2.connect(**DB_PARAMS)
cur = conn.cursor(cursor_factory=RealDictCursor)

cur.execute("SELECT id, name, filename, created_at, metadata FROM datasets ORDER BY created_at")
datasets = cur.fetchall()

print("Available datasets:")
print("=" * 80)
for ds in datasets:
    print(f"  ID:       {ds['id']}")
    print(f"  Name:     {ds['name']}")
    print(f"  File:     {ds['filename']}")
    print(f"  Created:  {ds['created_at']}")
    print(f"  Metadata: {ds['metadata']}")
    print()

cur.close()
conn.close()

In [ ]:
# ── Set the dataset ID to generate COGs for ────────────────────────────
# Copy a UUID from the list above:
DATASET_ID = datasets[0]['id']  # Default to first dataset
# ────────────────────────────────────────────────────────────────────────

# Output directory uses dataset UUID
COG_BASE_DIR = Path.cwd().parent / "local_only" / "climate_gt" / str(DATASET_ID)
COG_BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Selected dataset: {DATASET_ID}")
print(f"COG output dir:   {COG_BASE_DIR}")

## Database Access Functions

In [ ]:
@contextmanager
def get_db_cursor():
    """Context manager for database cursor."""
    conn = psycopg2.connect(**DB_PARAMS)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        cur.close()
        conn.close()

def get_available_variables(dataset_id):
    """Get unique variables for a dataset."""
    with get_db_cursor() as cur:
        cur.execute(
            "SELECT DISTINCT variable FROM climate_gt WHERE dataset_id = %s ORDER BY variable",
            [str(dataset_id)]
        )
        return [row['variable'] for row in cur.fetchall()]

def get_available_times(dataset_id, variable):
    """Get unique times for a variable in a dataset."""
    with get_db_cursor() as cur:
        cur.execute(
            "SELECT DISTINCT time FROM climate_gt WHERE dataset_id = %s AND variable = %s ORDER BY time",
            [str(dataset_id), variable]
        )
        return [row['time'] for row in cur.fetchall()]

def get_grid_data(dataset_id, variable, time):
    """Load grid data for a dataset, variable, and time."""
    with get_db_cursor() as cur:
        cur.execute(
            "SELECT lat, lon, value FROM climate_gt WHERE dataset_id = %s AND variable = %s AND time::text LIKE %s ORDER BY lat, lon",
            [str(dataset_id), variable, f"{time}%"]
        )
        rows = cur.fetchall()
        if not rows:
            return None
        return {
            'lats': np.array([row['lat'] for row in rows]),
            'lons': np.array([row['lon'] for row in rows]),
            'values': np.array([row['value'] for row in rows]),
        }

def get_global_min_max(dataset_id, variable):
    """Get global min/max across all times for a variable in a dataset."""
    with get_db_cursor() as cur:
        cur.execute(
            "SELECT MIN(value) as min_val, MAX(value) as max_val FROM climate_gt WHERE dataset_id = %s AND variable = %s",
            [str(dataset_id), variable]
        )
        row = cur.fetchone()
        return float(row['min_val']), float(row['max_val'])

print("Database functions defined")

## COG Generation Functions

In [ ]:
def normalize_values(values, data_min, data_max):
    """Normalize values to [0,1], return (normalized, grayscale, alpha)."""
    nan_mask = np.isnan(values)
    alpha = np.where(nan_mask, 0, 255).astype(np.uint8)
    
    if data_max == data_min:
        normalized = np.full_like(values, 0.5, dtype=float)
        grayscale = np.full_like(values, 127, dtype=np.uint8)
    else:
        normalized = (values - data_min) / (data_max - data_min)
        normalized = np.clip(normalized, 0, 1)
        grayscale = (normalized * 255).astype(np.uint8)
    
    grayscale[nan_mask] = 0
    return normalized, grayscale, alpha

def rasterize_grid_data_native(lats, lons, values, bounds):
    """Rasterize grid data at native resolution."""
    unique_lats = np.sort(np.unique(lats))[::-1]  # descending
    unique_lons = np.sort(np.unique(lons))
    native_h, native_w = len(unique_lats), len(unique_lons)
    
    print(f"  - Native grid resolution: {native_w}x{native_h}")
    
    lat_to_idx = {lat: idx for idx, lat in enumerate(unique_lats)}
    lon_to_idx = {lon: idx for idx, lon in enumerate(unique_lons)}
    
    y_indices = np.array([lat_to_idx.get(lat, -1) for lat in lats])
    x_indices = np.array([lon_to_idx.get(lon, -1) for lon in lons])
    valid = (y_indices >= 0) & (x_indices >= 0) & ~np.isnan(values)
    
    pixels = np.full((native_h, native_w), np.nan, dtype=np.float32)
    pixels[y_indices[valid], x_indices[valid]] = values[valid]
    return pixels

def create_and_save_cog(variable, time, zoom_level, native_raster_wgs84, data_min, data_max, bounds_wgs84):
    """
    Create COG in Web Mercator (EPSG:3857) with 5 bands.
    Reprojects WGS84 -> Web Mercator via rasterio.warp.
    """
    native_h, native_w = native_raster_wgs84.shape
    target_w, target_h = ZOOM_SPECS[zoom_level]

    src_crs = CRS.from_epsg(4326)
    dst_crs = CRS.from_epsg(3857)

    # Half-pixel correction
    cell_x = (bounds_wgs84['east'] - bounds_wgs84['west']) / (native_w - 1)
    cell_y = (bounds_wgs84['north'] - bounds_wgs84['south']) / (native_h - 1)
    adj = {
        'west':  bounds_wgs84['west']  - cell_x / 2,
        'east':  bounds_wgs84['east']  + cell_x / 2,
        'south': bounds_wgs84['south'] - cell_y / 2,
        'north': bounds_wgs84['north'] + cell_y / 2,
    }

    src_transform = from_bounds(adj['west'], adj['south'], adj['east'], adj['north'], native_w, native_h)
    dst_transform, _, _ = calculate_default_transform(
        src_crs, dst_crs, native_w, native_h,
        left=adj['west'], bottom=adj['south'], right=adj['east'], top=adj['north'],
        dst_width=target_w, dst_height=target_h,
    )

    # Reproject raw float data WGS84 -> Web Mercator
    NODATA = np.float32(-9999.0)
    src_float = native_raster_wgs84.astype(np.float32)
    src_float[np.isnan(src_float)] = NODATA

    reprojected = np.full((target_h, target_w), NODATA, dtype=np.float32)
    reproject(
        source=src_float, destination=reprojected,
        src_transform=src_transform, src_crs=src_crs,
        dst_transform=dst_transform, dst_crs=dst_crs,
        src_nodata=NODATA, dst_nodata=NODATA,
        resampling=Resampling.bilinear,
    )
    del src_float
    reprojected[reprojected == NODATA] = np.nan

    # Colorise
    normalized, grayscale, alpha = normalize_values(reprojected, data_min, data_max)
    del reprojected
    rgb_band = apply_colormap(normalized, COLORMAP_NAME)
    del normalized

    # Write GeoTIFF
    output_dir = COG_BASE_DIR / variable / str(time)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"z{zoom_level}.tif"

    profile = {
        'driver': 'GTiff', 'height': target_h, 'width': target_w,
        'count': 5, 'dtype': rasterio.uint8,
        'crs': dst_crs, 'transform': dst_transform,
        'compress': 'lzw', 'TILED': 'YES', 'BLOCKXSIZE': 512, 'BLOCKYSIZE': 512,
    }

    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(rgb_band[:, :, 0], 1)
        dst.write(rgb_band[:, :, 1], 2)
        dst.write(rgb_band[:, :, 2], 3)
        del rgb_band
        dst.write(grayscale, 4)
        del grayscale
        dst.write(alpha, 5)
        del alpha
        dst.update_tags(
            VARIABLE=variable, TIME=str(time), ZOOM_LEVEL=str(zoom_level),
            DATA_MIN=str(data_min), DATA_MAX=str(data_max),
            COLORMAP_TYPE=COLORMAP_NAME, CRS='EPSG:3857',
            DATASET_ID=str(DATASET_ID),
            BOUNDS_WGS84=f"[{adj['west']}, {adj['south']}, {adj['east']}, {adj['north']}]",
        )

    return output_path

print("COG creation function defined")

## Select Variable & Times

In [ ]:
available_variables = get_available_variables(DATASET_ID)
print(f"Available variables: {available_variables}")

selected_variable = available_variables[0]
print(f"Selected variable: {selected_variable}")

In [ ]:
available_times = get_available_times(DATASET_ID, selected_variable)
print(f"Available times: {len(available_times)} entries")
print(f"First 5: {available_times[:5]}")
print(f"Last 5:  {available_times[-5:]}")

## Compute Global Min/Max

In [ ]:
global_min, global_max = get_global_min_max(DATASET_ID, selected_variable)
print(f"Global range: {global_min:.2f} to {global_max:.2f}")

## Generate COGs (All Times, z0-z5)

In [ ]:
print(f"Generating COGs for dataset {DATASET_ID}")
print(f"Variable: {selected_variable} | Global range: {global_min:.2f} to {global_max:.2f}")
print(f"Times to process: {len(available_times)}")
print("=" * 60)

generated_files = []

for time_idx, time_val in enumerate(available_times):
    time_str = str(time_val)[:10]
    print(f"\n[{time_idx + 1}/{len(available_times)}] Processing {time_str}...")
    
    try:
        grid_data = get_grid_data(DATASET_ID, selected_variable, time_str)
        if grid_data is None:
            print(f"  No data for {time_str}, skipping")
            continue
        
        lats = grid_data['lats']
        lons = grid_data['lons']
        values = grid_data['values']
        
        print(f"  - Rasterizing at native resolution...")
        native_raster = rasterize_grid_data_native(lats, lons, values, AUSTRALIA_BOUNDS_WGS84)
        
        for zoom_level in sorted(ZOOM_SPECS.keys()):
            try:
                output_path = create_and_save_cog(
                    selected_variable, time_str, zoom_level,
                    native_raster, global_min, global_max, AUSTRALIA_BOUNDS_WGS84,
                )
                generated_files.append(output_path)
            except Exception as e:
                print(f"    ERROR at z{zoom_level}: {e}")
        
        print(f"  Generated 6 COGs (z0-z5) for {time_str}")
        del native_raster
        
    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "=" * 60)
print(f"Generated {len(generated_files)} COG files total")
print(f"Coverage: {len(generated_files) // 6} time slices x 6 zoom levels")

## Verification

In [ ]:
print("Generated Files Summary:")
print("=" * 60)

first_time = str(available_times[0])[:10]
output_dir = COG_BASE_DIR / selected_variable / first_time
if output_dir.exists():
    files = sorted(output_dir.glob('z*.tif'))
    total_size_mb = 0
    for f in files:
        size_mb = f.stat().st_size / (1024**2)
        total_size_mb += size_mb
        print(f"{f.name:12} {size_mb:8.2f} MB")
    print("=" * 60)
    print(f"{'Total':12} {total_size_mb:8.2f} MB")
    print(f"Location: {output_dir}")
else:
    print("Output directory not found!")

In [ ]:
# Verify first COG
print("Verifying first COG (z0)...")
test_file = COG_BASE_DIR / selected_variable / str(available_times[0])[:10] / "z0.tif"

if test_file.exists():
    with rasterio.open(test_file) as src:
        print(f"CRS:    {src.crs}")
        print(f"Size:   {src.width}x{src.height}")
        print(f"Bands:  {src.count}")
        print(f"Dtype:  {src.dtypes[0]}")
        print(f"Bounds: {src.bounds}")
        print(f"Tags:")
        for key, value in src.tags().items():
            print(f"  {key}: {value}")
else:
    print(f"Test file not found: {test_file}")